In [0]:
# ============================================================
# CELL 1 — IMPORT REQUIRED PYSPARK FUNCTIONS
# ============================================================

# Import PySpark SQL functions.
# We will use these functions for:
# - generating record IDs
# - handling NULL values
# - adding ingestion metadata
# - performing transformations
from pyspark.sql import functions as F


# Confirm that the required PySpark functions are available.
print("PySpark functions imported successfully.")

Read the pipeline configuration

In [0]:
# ============================================================
# CELL 2 — READ PIPELINE METADATA
# ============================================================

# Read the pipeline metadata table from Unity Catalog.
# This table contains the configuration required by our
# metadata-driven ingestion framework.
metadata_df = spark.table(
    "workspace.nyc_taxi_audit.pipeline_metadata"
)


# Select only active pipeline configurations.
# inactive pipelines must not be processed.
active_metadata_df = (
    metadata_df
    .filter(F.col("active_flag") == True)
)


# Display the active configurations.
# This is only for verification during development.
display(active_metadata_df)

Convert active metadata to a Python configuration

In [0]:
# ============================================================
# CELL 3 — CREATE PIPELINE CONFIGURATION
# ============================================================

# Count the active pipeline configurations.
# Our current framework is intentionally designed to process
# one active pipeline at a time.
active_pipeline_count = active_metadata_df.count()


# Stop execution if no active pipeline is configured.
if active_pipeline_count == 0:

    raise ValueError(
        "No active pipeline configuration found."
    )


# Stop execution if multiple active pipelines are found.
# We will support multiple pipelines later after the
# single-pipeline flow is working correctly.
if active_pipeline_count > 1:

    raise ValueError(
        "Multiple active pipelines found. "
        "Current Bronze implementation expects one "
        "active pipeline at a time."
    )


# Retrieve the single active configuration.
pipeline_config_row = (
    active_metadata_df.first()
)


# Convert the Spark Row into a Python dictionary.
# This lets the ingestion code access configuration values
# using dictionary keys.
pipeline_config = pipeline_config_row.asDict()


# Display the selected pipeline name.
print(
    "Pipeline selected:",
    pipeline_config["pipeline_name"]
)


# Display the complete pipeline configuration
# for development-time verification.
print(
    "Pipeline configuration:",
    pipeline_config
)

Prepare the ingestion configuration

In [0]:
# ============================================================
# CELL 4 — PREPARE INGESTION CONFIGURATION
# ============================================================

# Read the comma-separated list of source columns
# that we stored in pipeline_metadata.
record_hash_columns_string = (
    pipeline_config["record_hash_columns"]
)


# Convert the comma-separated string into a Python list.
#
# Example:
# "VendorID,trip_distance,fare_amount"
#
# becomes:
# ["VendorID", "trip_distance", "fare_amount"]
record_hash_columns = [
    column_name.strip()
    for column_name in record_hash_columns_string.split(",")
    if column_name.strip()
]


# Validate that at least one hash column is configured.
# Without hash columns, we cannot generate a deterministic
# record_id for the Bronze data.
if not record_hash_columns:

    raise ValueError(
        "No record hash columns configured "
        "for the pipeline."
    )


# Build the fully qualified Unity Catalog target table name
# using the catalog, schema, and table values from metadata.
#
# Example:
# workspace.nyc_taxi_bronze.yellow_taxi
target_table_name = (
    f"{pipeline_config['target_catalog']}."
    f"{pipeline_config['target_schema']}."
    f"{pipeline_config['target_table']}"
)


# Display the values prepared for the ingestion engine.
print(
    "Target table:",
    target_table_name
)

print(
    "Record hash columns:",
    record_hash_columns
)

Discover matching source files

In [0]:
# ============================================================
# CELL 5 — DISCOVER SOURCE FILES
# ============================================================

# Read the source directory from the pipeline configuration.
# This value comes from pipeline_metadata.
source_path = pipeline_config["source_path"]


# Read the file pattern from the pipeline configuration.
# Example:
# yellow_tripdata_*.parquet
source_file_pattern = pipeline_config["source_file_pattern"]


# List all files available in the configured source directory.
# dbutils.fs.ls() returns metadata about each file.
source_files = dbutils.fs.ls(source_path)


# Import Python's fnmatch module.
# fnmatch allows us to compare filenames against wildcard
# patterns such as yellow_tripdata_*.parquet.
import fnmatch


# Keep only files whose names match the configured pattern.
matching_files = [
    file_info
    for file_info in source_files
    if fnmatch.fnmatch(
        file_info.name,
        source_file_pattern
    )
]


# Make sure the configured pattern actually found files.
# A missing file should not silently result in a successful run.
if not matching_files:

    raise FileNotFoundError(
        f"No source files found in {source_path} "
        f"matching pattern {source_file_pattern}"
    )


# Display how many matching files were discovered.
print(
    f"Matching source files found: {len(matching_files)}"
)


# Display each matching file and its size in bytes.
for file_info in matching_files:

    print(
        f"File: {file_info.name} | "
        f"Size: {file_info.size} bytes"
    )

Create a clean list of source file paths

In [0]:
# ============================================================
# CELL 6 — CREATE SOURCE FILE PATH LIST
# ============================================================

# Convert the discovered file metadata into a simple list
# containing only the complete source file paths.
#
# This makes the files easier for the ingestion framework
# to process one by one.
source_file_paths = [
    file_info.path
    for file_info in matching_files
]


# Validate that at least one source file path exists.
if not source_file_paths:

    # Stop execution if no source files are available.
    raise FileNotFoundError(
        "No source file paths were discovered."
    )


# Display the number of files available for processing.
print(
    f"Source files ready for processing: "
    f"{len(source_file_paths)}"
)


# Display every source file path.
for file_path in source_file_paths:

    print(
        f"Source file: {file_path}"
    )

Check which files were already processed

In [0]:
# ============================================================
# CELL 7 — CHECK ALREADY PROCESSED FILES
# ============================================================

# Read the pipeline ID from our metadata configuration.
# This identifies which pipeline owns the source files.
pipeline_id = pipeline_config["pipeline_id"]


# Read the pipeline run audit table.
# This table stores information about previous executions.
run_log_df = spark.table(
    "workspace.nyc_taxi_audit.pipeline_run_log"
)


# Select previously processed source files for this pipeline.
#
# We only consider files from successful or partially successful
# runs as previously processed.
processed_files_df = (
    run_log_df
    .filter(
        (F.col("pipeline_id") == pipeline_id)
        & (
            F.col("status").isin(
                ["SUCCESS", "PARTIAL"]
            )
        )
        & F.col("source_file").isNotNull()
    )
    .select("source_file")
    .distinct()
)


# Collect the processed file paths into a Python set.
#
# A set gives us fast membership checks when we examine
# newly discovered source files.
processed_file_paths = {
    row["source_file"]
    for row in processed_files_df.collect()
}


# Identify files that have NOT been successfully processed yet.
new_source_file_paths = [
    file_path
    for file_path in source_file_paths
    if file_path not in processed_file_paths
]


# Display the processing summary.
print(
    f"Total discovered files: {len(source_file_paths)}"
)

print(
    f"Previously processed files: "
    f"{len(processed_file_paths)}"
)

print(
    f"New files ready for processing: "
    f"{len(new_source_file_paths)}"
)


# Display the files that still need to be processed.
for file_path in new_source_file_paths:

    print(
        f"New source file: {file_path}"
    )

Generate the pipeline run ID

In [0]:
# ============================================================
# CELL 8 — CREATE PIPELINE RUN ID
# ============================================================

# Import Python's UUID library.
# UUID gives us a practically unique identifier for each
# pipeline execution.
import uuid


# Generate a unique identifier for this pipeline run.
run_id = str(uuid.uuid4())


# Capture the pipeline ID from our metadata configuration.
pipeline_id = pipeline_config["pipeline_id"]


# Capture the pipeline name from our metadata configuration.
pipeline_name = pipeline_config["pipeline_name"]


# Define how this run was started.
# For our current development execution, we use MANUAL.
trigger_type = "MANUAL"


# Display the generated run information.
print(
    f"Pipeline run created: {run_id}"
)

print(
    f"Pipeline ID: {pipeline_id}"
)

print(
    f"Pipeline name: {pipeline_name}"
)

print(
    f"Trigger type: {trigger_type}"
)

Write the initial RUNNING audit record

In [0]:
# ============================================================
# CELL 9 — WRITE INITIAL RUNNING AUDIT RECORD
# ============================================================

# Capture the current timestamp.
# This represents the moment when the pipeline execution starts.
run_start_time = spark.sql(
    "SELECT current_timestamp() AS current_time"
).collect()[0]["current_time"]


# Get the first source file that is ready for processing.
# At this stage of development, we expect one new file.
source_file = new_source_file_paths[0]


# Create a Python list containing the values
# required for our pipeline_run_log record.
run_log_data = [
    (
        # Unique ID for this pipeline execution.
        run_id,

        # Pipeline configuration identifier.
        pipeline_id,

        # Human-readable pipeline name.
        pipeline_name,

        # How this execution was started.
        trigger_type,

        # Source file being processed.
        source_file,

        # Pipeline start time.
        run_start_time,

        # Pipeline has not finished yet.
        None,

        # Initial execution status.
        "RUNNING",

        # Records read will be populated after ingestion.
        0,

        # Records written will be populated after ingestion.
        0,

        # Rejected records will be populated after DQ processing.
        0,

        # Watermark is not used in our FILE-based strategy.
        None,

        # No error at startup.
        None,

        # Databricks job run ID will be added later
        # when we execute through a Workflow.
        None,

        # Audit record creation timestamp.
        run_start_time
    )
]


# Define the schema for the RUNNING audit record.
run_log_schema = """

    run_id STRING,
    pipeline_id STRING,
    pipeline_name STRING,
    trigger_type STRING,
    source_file STRING,
    run_start_time TIMESTAMP,
    run_end_time TIMESTAMP,
    status STRING,
    records_read BIGINT,
    records_written BIGINT,
    records_rejected BIGINT,
    watermark_value STRING,
    error_message STRING,
    job_run_id STRING,
    created_at TIMESTAMP
"""


# Create a temporary DataFrame containing the RUNNING record.
run_log_df = spark.createDataFrame(
    run_log_data,
    run_log_schema
)


# Append the RUNNING record to our pipeline audit table.
run_log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "workspace.nyc_taxi_audit.pipeline_run_log"
    )


# Confirm that the audit record was written.
print(
    f"Pipeline run {run_id} recorded with status RUNNING."
)

Read the source file

In [0]:
# ============================================================
# CELL 10 — READ NEW SOURCE FILE
# ============================================================

# Read the source format from our metadata configuration.
source_format = pipeline_config["source_format"]


# Read the source file path selected for this pipeline run.
source_file = new_source_file_paths[0]


# Make sure the configured source format is supported.
if source_format.lower() != "parquet":

    # Stop execution if the metadata specifies
    # a format that our current framework doesn't support.
    raise ValueError(
        f"Unsupported source format: {source_format}"
    )


# Read the Parquet source file into a Spark DataFrame.
#
# The path comes from metadata-driven file discovery.
# Nothing about the January filename is hardcoded here.
source_df = spark.read.parquet(
    source_file
)


# Count the records that were read from the source file.
records_read = source_df.count()


# Display basic ingestion information.
print(
    f"Source file successfully read: {source_file}"
)

print(
    f"Records read: {records_read}"
)


# Display the source schema so we can verify
# what was actually loaded.
source_df.printSchema()

Generate record_id

In [0]:
# ============================================================
# CELL 11 — GENERATE DETERMINISTIC RECORD ID
# ============================================================

# Use the hash columns defined in our pipeline metadata.
# These columns represent the source record content.
#
# We do not include pipeline-generated columns such as:
# run_id, ingestion_timestamp, or source_file.
# Those values belong to the ingestion metadata layer.
record_hash_columns = [
    column_name.strip()
    for column_name in
    pipeline_config["record_hash_columns"].split(",")
    if column_name.strip()
]


# Generate a deterministic SHA-256 hash for every source record.
#
# concat_ws("||", ...) combines the configured source values.
#
# coalesce(..., "<NULL>") gives NULL values a consistent
# representation so that the same source record always
# generates the same hash.
#
# sha2(..., 256) creates the final technical record identifier.
bronze_df = (
    source_df
    .withColumn(
        "record_id",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(column_name).cast("string"),
                        F.lit("<NULL>")
                    )
                    for column_name in record_hash_columns
                ]
            ),
            256
        )
    )
)


# Display a small sample so we can verify the new record_id.
display(
    bronze_df.select(
        "record_id",
        *record_hash_columns
    ).limit(10)
)

Add ingestion metadata

In [0]:
# ============================================================
# CELL 12 — ADD INGESTION METADATA
# ============================================================

# Add the unique pipeline run ID to every Bronze record.
#
# This lets us trace every record back to the exact
# pipeline execution that processed it.
bronze_df = bronze_df.withColumn(
    "run_id",
    F.lit(run_id)
)


# Add the source file path to every Bronze record.
#
# This gives us source-level lineage and helps us identify
# which physical source file produced a record.
bronze_df = bronze_df.withColumn(
    "source_file",
    F.lit(source_file)
)


# Add the ingestion timestamp.
#
# This represents when our Databricks pipeline ingested
# the record into the Bronze layer.
bronze_df = bronze_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)


# Display the new metadata columns along with record_id.
display(
    bronze_df.select(
        "record_id",
        "run_id",
        "source_file",
        "ingestion_timestamp"
    ).limit(10)
)

Validate record_id uniqueness

In [0]:
# ============================================================
# CELL 13 — VALIDATE RECORD ID UNIQUENESS
# ============================================================

# Count the total number of records currently being processed.
# This should match the number of records read from the source.
total_bronze_records = bronze_df.count()


# Count the number of distinct record IDs.
# Every source record should have its own deterministic ID.
distinct_record_ids = (
    bronze_df
    .select("record_id")
    .distinct()
    .count()
)


# Calculate how many records share the same record_id.
duplicate_record_ids = (
    total_bronze_records - distinct_record_ids
)


# Display the validation results.
print(
    f"Total records: {total_bronze_records}"
)

print(
    f"Distinct record IDs: {distinct_record_ids}"
)

print(
    f"Duplicate record IDs: {duplicate_record_ids}"
)


# Fail the ingestion if duplicate technical record IDs exist.
#
# We do this BEFORE writing to Bronze so that a broken
# record-identification strategy cannot corrupt our target.
if duplicate_record_ids > 0:

    raise ValueError(
        f"Duplicate record_id values detected: "
        f"{duplicate_record_ids}"
    )


# Confirm successful validation.
print(
    "Record ID uniqueness validation successful."
)

Check the Bronze target table

In [0]:
# ============================================================
# CELL 14 — CHECK BRONZE TARGET TABLE
# ============================================================

# Check whether our metadata-driven Bronze target table
# already exists in Unity Catalog.
target_table_exists = spark.catalog.tableExists(
    target_table_name
)


# Display the result of the table existence check.
print(
    f"Target table: {target_table_name}"
)

print(
    f"Target table exists: {target_table_exists}"
)

Write the first Bronze Delta table

In [0]:
# ============================================================
# CELL 15 — WRITE INITIAL BRONZE DELTA TABLE
# ============================================================

# Write the prepared Bronze DataFrame as a Delta table.
#
# Since the target table does not exist yet, this is our
# initial load.
#
# We use "overwrite" ONLY for this first table creation.
# We will NOT use overwrite for future incremental loads.
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        target_table_name
    )


# Confirm that the Bronze table was created successfully.
print(
    f"Bronze Delta table created successfully: "
    f"{target_table_name}"
)

Verify Bronze table

In [0]:
# ============================================================
# CELL 16 — VERIFY BRONZE DELTA TABLE
# ============================================================

# Read the Bronze Delta table that we just created.
# The table name comes from our metadata-driven configuration.
bronze_table_df = spark.table(
    target_table_name
)


# Count the records stored in the Bronze table.
bronze_record_count = bronze_table_df.count()


# Display the number of records written.
print(
    f"Bronze records written: {bronze_record_count}"
)


# Compare the Bronze record count with the number of
# records that we originally read from the source.
print(
    f"Source records read: {records_read}"
)


# Validate that the counts match.
#
# For this initial load, every source record should have been
# written to Bronze.
if bronze_record_count != records_read:

    raise ValueError(
        f"Bronze record count mismatch. "
        f"Source={records_read}, "
        f"Bronze={bronze_record_count}"
    )


# Confirm successful validation.
print(
    "Bronze record count validation successful."
)


# Display a small sample of the Bronze data.
display(
    bronze_table_df.limit(10)
)

Validate Bronze Delta properties

In [0]:
# ============================================================
# CELL 17 — VALIDATE BRONZE DELTA TABLE
# ============================================================

# Check the physical format of the Bronze target table.
# We expect the table to be stored using Delta Lake.
table_detail_df = spark.sql(
    f"DESCRIBE DETAIL {target_table_name}"
)


# Display the table details.
display(table_detail_df)


# Read the schema of the Bronze table.
# This allows us to confirm that both source columns
# and ingestion metadata columns were written.
bronze_table_df.printSchema()

Update pipeline_run_log to SUCCESS

In [0]:
# ============================================================
# CELL 18 — COMPLETE PIPELINE RUN AUDIT
# ============================================================

# Capture the pipeline completion timestamp.
# This represents when the Bronze ingestion finished successfully.
run_end_time = spark.sql(
    "SELECT current_timestamp() AS current_time"
).collect()[0]["current_time"]


# Store the number of records rejected during this stage.
# We have not implemented the Silver DQ/quarantine framework yet,
# so no records have been rejected at this point.
records_rejected = 0


# Store the number of records successfully written to Bronze.
records_written = bronze_record_count


# Update the existing RUNNING audit record using the current run_id.
#
# We update the same execution record instead of creating
# another SUCCESS record for the same run.
spark.sql(f"""
UPDATE workspace.nyc_taxi_audit.pipeline_run_log

SET

    -- Record the completion timestamp.
    run_end_time = TIMESTAMP('{run_end_time}'),

    -- Mark the pipeline execution as successful.
    status = 'SUCCESS',

    -- Record how many source records were read.
    records_read = {records_read},

    -- Record how many records were written to Bronze.
    records_written = {records_written},

    -- No records have been rejected at this stage.
    records_rejected = {records_rejected}

WHERE run_id = '{run_id}'
""")


# Confirm that the audit update completed.
print(
    f"Pipeline run {run_id} updated to SUCCESS."
)

Verify the final pipeline audit record

In [0]:
# ============================================================
# CELL 19 — VERIFY PIPELINE RUN AUDIT
# ============================================================

# Read the audit record for the current pipeline execution.
pipeline_run_audit_df = (
    spark.table(
        "workspace.nyc_taxi_audit.pipeline_run_log"
    )
    .filter(
        F.col("run_id") == run_id
    )
)


# Display the current run audit record.
display(pipeline_run_audit_df)

Verify the current Bronze count

In [0]:
# ============================================================
# CELL 20 — CAPTURE CURRENT BRONZE RECORD COUNT
# ============================================================

# Read the current Bronze table.
# The target table name comes from our metadata configuration.
current_bronze_df = spark.table(
    target_table_name
)


# Count the records currently stored in Bronze.
current_bronze_count = current_bronze_df.count()


# Display the current Bronze record count.
print(
    f"Current Bronze record count: "
    f"{current_bronze_count}"
)


# Store this value as our baseline.
# We will compare it against the count after a rerun.
bronze_count_before_rerun = current_bronze_count


# Confirm that the baseline count has been captured.
print(
    "Bronze baseline count captured successfully."
)

Determine files eligible for processing

In [0]:
# ============================================================
# CELL 21 — IDENTIFY FILES ELIGIBLE FOR PROCESSING
# ============================================================

# Read the pipeline ID from the metadata configuration.
# We use this to look only at audit records belonging
# to the current pipeline.
pipeline_id = pipeline_config["pipeline_id"]


# Read the pipeline run audit table.
# This contains the history of previous pipeline executions.
run_log_df = spark.table(
    "workspace.nyc_taxi_audit.pipeline_run_log"
)


# Find source files that were already processed successfully.
#
# We only treat SUCCESS as fully processed here.
# A PARTIAL run may require additional handling later.
processed_files_df = (
    run_log_df
    .filter(
        (F.col("pipeline_id") == pipeline_id)
        & (F.col("status") == "SUCCESS")
        & F.col("source_file").isNotNull()
    )
    .select(
        "source_file"
    )
    .distinct()
)


# Convert the processed source-file list into a Python set.
#
# A set allows us to efficiently check whether a discovered
# source file has already been processed.
processed_file_paths = {
    row["source_file"]
    for row in processed_files_df.collect()
}


# Identify only source files that have NOT been successfully
# processed by this pipeline.
eligible_file_paths = [
    file_path
    for file_path in source_file_paths
    if file_path not in processed_file_paths
]


# Display the idempotency decision.
print(
    f"Discovered source files: "
    f"{len(source_file_paths)}"
)

print(
    f"Previously successful files: "
    f"{len(processed_file_paths)}"
)

print(
    f"Files eligible for processing: "
    f"{len(eligible_file_paths)}"
)


# Display the files that are eligible for processing.
for file_path in eligible_file_paths:

    print(
        f"Eligible file: {file_path}"
    )

Explicitly verify that rerun would be skipped

In [0]:
# ============================================================
# CELL 22 — CONFIRM IDEMPOTENCY DECISION
# ============================================================

# Check whether there are any files that still need processing.
# If the list is empty, all discovered files have already been
# successfully processed.
if len(eligible_file_paths) == 0:

    # Display a clear message indicating that the pipeline
    # will skip the discovered files.
    print(
        "IDEMPOTENCY CHECK PASSED: "
        "All discovered source files were already processed successfully."
    )

    # Display the files that were skipped.
    print(
        f"Skipped files: {len(source_file_paths)}"
    )


# If at least one new file exists, show that processing is required.
else:

    print(
        f"New files require processing: "
        f"{len(eligible_file_paths)}"
    )


# Display the Bronze baseline count so that we have
# a reference point before any future processing occurs.
print(
    f"Bronze record count before processing: "
    f"{bronze_count_before_rerun}"
)

Check the file registry before processing

In [0]:
# ============================================================
# CELL 23 — CHECK FILE INGESTION HISTORY
# ============================================================

# Read the file-level ingestion audit table.
# This table maintains the processing history of individual
# source files.
file_ingestion_log_df = spark.table(
    "workspace.nyc_taxi_audit.file_ingestion_log"
)


# Select files that were already processed successfully
# by the current pipeline.
successful_files_df = (
    file_ingestion_log_df
    .filter(
        (F.col("pipeline_id") == pipeline_id)
        & (F.col("status") == "SUCCESS")
    )
    .select(
        "source_file"
    )
    .distinct()
)


# Convert the successful file paths into a Python set.
# A set gives us efficient membership checks.
successful_file_paths = {
    row["source_file"]
    for row in successful_files_df.collect()
}


# Compare the discovered files against the file-level
# processing history.
new_file_paths = [
    file_path
    for file_path in source_file_paths
    if file_path not in successful_file_paths
]


# Display the file-processing summary.
print(
    f"Discovered files: {len(source_file_paths)}"
)

print(
    f"Previously successful files: "
    f"{len(successful_file_paths)}"
)

print(
    f"New files requiring processing: "
    f"{len(new_file_paths)}"
)


# Display any files that are still eligible for processing.
for file_path in new_file_paths:

    print(
        f"New file: {file_path}"
    )

Backfill the existing successful file

In [0]:
# ============================================================
# CELL 24 — BACKFILL FILE INGESTION HISTORY
# ============================================================

# Capture the current timestamp.
# We use this as the time when the file-level audit record
# is being created.
file_audit_time = spark.sql(
    "SELECT current_timestamp() AS current_time"
).collect()[0]["current_time"]


# Get the file information for our already-processed source file.
# The first matching file is our January 2025 Parquet file.
processed_file_info = matching_files[0]


# Extract the complete source file path.
processed_source_file = processed_file_info.path


# Extract only the file name from the path.
processed_source_file_name = processed_file_info.name


# Extract the file size in bytes.
processed_file_size = processed_file_info.size


# Create a DataFrame containing one SUCCESS record
# for the January file that we already processed.
file_audit_data = [
    (
        # The run_id from the successful Bronze execution.
        run_id,

        # Pipeline configuration identifier.
        pipeline_id,

        # Human-readable pipeline name.
        pipeline_name,

        # Complete source file path.
        processed_source_file,

        # File name only.
        processed_source_file_name,

        # Source file size in bytes.
        processed_file_size,

        # Time when the file was discovered.
        file_audit_time,

        # Processing start time.
        run_start_time,

        # Processing completion time.
        run_end_time,

        # File processing status.
        "SUCCESS",

        # Number of records read.
        records_read,

        # Number of records written.
        records_written,

        # Number of records rejected.
        records_rejected,

        # No error because processing succeeded.
        None,

        # Audit record creation timestamp.
        file_audit_time
    )
]


# Define the schema for the file ingestion audit record.
file_audit_schema = """
    run_id STRING,
    pipeline_id STRING,
    pipeline_name STRING,
    source_file STRING,
    source_file_name STRING,
    file_size_bytes BIGINT,
    discovered_at TIMESTAMP,
    processing_start_time TIMESTAMP,
    processing_end_time TIMESTAMP,
    status STRING,
    records_read BIGINT,
    records_written BIGINT,
    records_rejected BIGINT,
    error_message STRING,
    created_at TIMESTAMP
"""


# Convert the Python record into a Spark DataFrame.
file_audit_df = spark.createDataFrame(
    file_audit_data,
    file_audit_schema
)


# Write the file-level SUCCESS record into our audit table.
file_audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "workspace.nyc_taxi_audit.file_ingestion_log"
    )


# Confirm that the file history was recorded.
print(
    f"File ingestion history recorded successfully: "
    f"{processed_source_file_name}"
)

Re-check file idempotency

In [0]:
# ============================================================
# CELL 25 — RE-CHECK FILE IDEMPOTENCY
# ============================================================

# Read the file-level ingestion history.
# This table tells us which source files have already
# completed successfully.
file_ingestion_log_df = spark.table(
    "workspace.nyc_taxi_audit.file_ingestion_log"
)


# Find all files that have already completed successfully
# for the current pipeline.
successful_files_df = (
    file_ingestion_log_df
    .filter(
        (F.col("pipeline_id") == pipeline_id)
        & (F.col("status") == "SUCCESS")
    )
    .select(
        "source_file"
    )
    .distinct()
)


# Convert the successful source files into a Python set.
# A set makes it efficient to check whether a source file
# has already been processed.
successful_file_paths = {
    row["source_file"]
    for row in successful_files_df.collect()
}


# Compare all discovered files against the successful
# file history.
new_file_paths = [
    file_path
    for file_path in source_file_paths
    if file_path not in successful_file_paths
]


# Display the final idempotency decision.
print(
    f"Discovered files: {len(source_file_paths)}"
)

print(
    f"Previously successful files: "
    f"{len(successful_file_paths)}"
)

print(
    f"Files requiring processing: "
    f"{len(new_file_paths)}"
)


# Display any files that still need processing.
for file_path in new_file_paths:

    print(
        f"New file: {file_path}"
    )

Inspect file ingestion history

In [0]:
# ============================================================
# CELL 26 — VERIFY FILE INGESTION HISTORY
# ============================================================

# Read the file-level ingestion audit table.
file_history_df = spark.table(
    "workspace.nyc_taxi_audit.file_ingestion_log"
)


# Filter the history for the current pipeline.
current_pipeline_file_history_df = (
    file_history_df
    .filter(
        F.col("pipeline_id") == pipeline_id
    )
)


# Display the file-processing history.
# This lets us verify that the January file was recorded
# as successfully processed.
display(
    current_pipeline_file_history_df
)

Separate new files from already processed files

In [0]:
# ============================================================
# CELL 27 — BUILD NEW FILE PROCESSING LIST
# ============================================================

# Read the file-level ingestion history.
# This table tells us which source files have already
# completed successfully.
file_ingestion_log_df = spark.table(
    "workspace.nyc_taxi_audit.file_ingestion_log"
)


# Find files that have already completed successfully
# for the current pipeline.
successful_files_df = (
    file_ingestion_log_df
    .filter(
        (F.col("pipeline_id") == pipeline_id)
        & (F.col("status") == "SUCCESS")
    )
    .select(
        "source_file"
    )
    .distinct()
)


# Convert the successful file paths into a Python set.
successful_file_paths = {
    row["source_file"]
    for row in successful_files_df.collect()
}


# Keep only discovered files that have never completed
# successfully before.
new_file_paths = [
    file_path
    for file_path in source_file_paths
    if file_path not in successful_file_paths
]


# Display the final processing decision.
print(
    f"Discovered files: {len(source_file_paths)}"
)

print(
    f"Successfully processed files: "
    f"{len(successful_file_paths)}"
)

print(
    f"New files requiring processing: "
    f"{len(new_file_paths)}"
)


# Display the new files.
for file_path in new_file_paths:

    print(
        f"New file: {file_path}"
    )

Create the file-processing summary

In [0]:
# ============================================================
# CELL 28 — CREATE FILE PROCESSING SUMMARY
# ============================================================

# Create a summary of the files discovered during this run.
# This gives us a simple operational view before processing.
file_processing_summary = {
    
    # Total number of files discovered from the source location.
    "discovered_files": len(source_file_paths),

    # Number of files that were already processed successfully.
    "already_processed_files": len(successful_file_paths),

    # Number of files that still need processing.
    "new_files": len(new_file_paths)
}


# Display the file-processing summary.
print(
    "File processing summary:"
)

print(
    f"Discovered files: "
    f"{file_processing_summary['discovered_files']}"
)

print(
    f"Already processed files: "
    f"{file_processing_summary['already_processed_files']}"
)

print(
    f"New files: "
    f"{file_processing_summary['new_files']}"
)

Record a no-op rerun

In [0]:
# ============================================================
# CELL 29 — RECORD NO-OP RERUN
# ============================================================

# Check whether there are any new source files to process.
if len(new_file_paths) == 0:

    # Generate a new run ID for this new pipeline execution.
    # This is different from the previous successful run.
    rerun_id = str(uuid.uuid4())


    # Capture the start time of this new execution.
    rerun_start_time = spark.sql(
        "SELECT current_timestamp() AS current_time"
    ).collect()[0]["current_time"]


    # Since there are no new files, this execution performs
    # no data processing.
    rerun_end_time = spark.sql(
        "SELECT current_timestamp() AS current_time"
    ).collect()[0]["current_time"]


    # Create an audit record for this no-op execution.
    rerun_log_data = [
        (
            # New execution identifier.
            rerun_id,

            # Current pipeline ID.
            pipeline_id,

            # Current pipeline name.
            pipeline_name,

            # This was a manual development rerun.
            "MANUAL",

            # No new file was processed.
            None,

            # Execution start time.
            rerun_start_time,

            # Execution end time.
            rerun_end_time,

            # The pipeline completed successfully,
            # but there was no new work to process.
            "SUCCESS",

            # No records were read.
            0,

            # No records were written.
            0,

            # No records were rejected.
            0,

            # No timestamp watermark is used.
            None,

            # No error occurred.
            None,

            # No Databricks job ID yet.
            None,

            # Audit creation timestamp.
            rerun_end_time
        )
    ]


    # Create a DataFrame containing the no-op audit record.
    rerun_log_df = spark.createDataFrame(
        rerun_log_data,
        run_log_schema
    )


    # Append the new execution record to pipeline_run_log.
    rerun_log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            "workspace.nyc_taxi_audit.pipeline_run_log"
        )


    # Display the result.
    print(
        "No new source files found."
    )

    print(
        f"Rerun {rerun_id} completed successfully "
        "with no data processing."
    )

    print(
        "Existing source files were skipped because "
        "they were already successfully ingested."
    )

else:

    # At least one new file exists, so actual ingestion
    # will be required.
    print(
        f"{len(new_file_paths)} new file(s) require processing."
    )

Create a reusable file-audit function

In [0]:
# ============================================================
# CELL 30 — CREATE FILE AUDIT FUNCTION
# ============================================================

# Define a reusable function that creates a file-level
# PROCESSING audit record.
#
# Instead of rewriting the same INSERT logic for every file,
# our ingestion framework can call this function whenever
# a new file starts processing.
def start_file_processing(
    run_id,
    pipeline_id,
    pipeline_name,
    file_info
):
    """
    Create a PROCESSING record in file_ingestion_log.

    Parameters:
        run_id:
            Unique ID of the current pipeline execution.

        pipeline_id:
            ID of the configured pipeline.

        pipeline_name:
            Human-readable pipeline name.

        file_info:
            Databricks FileInfo object containing
            file name, path, size, etc.
    """

    # Capture the time at which file processing starts.
    processing_start_time = spark.sql(
        "SELECT current_timestamp() AS current_time"
    ).collect()[0]["current_time"]


    # Extract the complete source file path.
    source_file = file_info.path


    # Extract only the file name.
    source_file_name = file_info.name


    # Extract the source file size in bytes.
    file_size_bytes = file_info.size


    # Create the file-level audit record.
    processing_record = [
        (
            # Pipeline execution ID.
            run_id,

            # Pipeline configuration ID.
            pipeline_id,

            # Pipeline name.
            pipeline_name,

            # Complete source file path.
            source_file,

            # File name.
            source_file_name,

            # File size.
            file_size_bytes,

            # File discovery time.
            processing_start_time,

            # File processing start time.
            processing_start_time,

            # Processing has not finished yet.
            None,

            # Current file status.
            "PROCESSING",

            # Records have not yet been read.
            0,

            # Records have not yet been written.
            0,

            # No records have been rejected yet.
            0,

            # No error at this point.
            None,

            # Audit record creation time.
            processing_start_time
        )
    ]


    # Use the same schema that we defined when creating
    # the file_ingestion_log table.
    processing_record_df = spark.createDataFrame(
        processing_record,
        file_audit_schema
    )


    # Append the PROCESSING record to the file audit table.
    processing_record_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            "workspace.nyc_taxi_audit.file_ingestion_log"
        )


    # Return the start time so the caller can use it
    # later when updating the audit record.
    return processing_start_time

Create the file completion function

In [0]:
# ============================================================
# CELL 31 — CREATE FILE SUCCESS AUDIT FUNCTION
# ============================================================

# Define a reusable function that marks a source file
# as successfully processed.
#
# This function updates the existing PROCESSING record
# for the file instead of creating a second SUCCESS record.
def complete_file_processing(
    run_id,
    source_file,
    records_read,
    records_written,
    records_rejected
):
    """
    Mark a file as successfully processed.

    Parameters:
        run_id:
            Unique identifier of the pipeline execution.

        source_file:
            Complete source file path.

        records_read:
            Number of records read from the source file.

        records_written:
            Number of records written to Bronze.

        records_rejected:
            Number of records rejected during processing.
    """

    # Capture the time when processing completed.
    processing_end_time = spark.sql(
        "SELECT current_timestamp() AS current_time"
    ).collect()[0]["current_time"]


    # Update the existing PROCESSING record.
    spark.sql(f"""
        UPDATE workspace.nyc_taxi_audit.file_ingestion_log

        SET

            -- Record when file processing completed.
            processing_end_time =
                TIMESTAMP('{processing_end_time}'),

            -- Mark the file as successfully processed.
            status = 'SUCCESS',

            -- Record the number of source records read.
            records_read = {records_read},

            -- Record the number of Bronze records written.
            records_written = {records_written},

            -- Record rejected records.
            records_rejected = {records_rejected},

            -- There was no processing error.
            error_message = NULL

        WHERE
            run_id = '{run_id}'
            AND source_file = '{source_file}'
            AND status = 'PROCESSING'
    """)


    # Confirm that the update was requested.
    print(
        f"File processing completed successfully: "
        f"{source_file}"
    )


    # Return the completion timestamp so the caller
    # can reuse it if necessary.
    return processing_end_time

Create the file failure audit function

In [0]:
# ============================================================
# CELL 32 — CREATE FILE FAILURE AUDIT FUNCTION
# ============================================================

# Define a reusable function that marks a source file
# as FAILED when a technical error occurs.
#
# This function updates the existing PROCESSING record
# instead of creating another row for the same execution/file.
def fail_file_processing(
    run_id,
    source_file,
    error_message,
    records_read=0,
    records_written=0,
    records_rejected=0
):
    """
    Mark a source file as FAILED.

    Parameters:
        run_id:
            Unique identifier of the pipeline execution.

        source_file:
            Complete source file path.

        error_message:
            Technical error that caused processing to fail.

        records_read:
            Number of records read before the failure.

        records_written:
            Number of records written before the failure.

        records_rejected:
            Number of rejected records before the failure.
    """

    # Capture the time when the failure occurred.
    processing_end_time = spark.sql(
        "SELECT current_timestamp() AS current_time"
    ).collect()[0]["current_time"]


    # Update the existing PROCESSING record.
    spark.sql(f"""
        UPDATE workspace.nyc_taxi_audit.file_ingestion_log

        SET

            -- Record when processing failed.
            processing_end_time =
                TIMESTAMP('{processing_end_time}'),

            -- Mark the file as failed.
            status = 'FAILED',

            -- Record records read before the failure.
            records_read = {records_read},

            -- Record records written before the failure.
            records_written = {records_written},

            -- Record rejected records before the failure.
            records_rejected = {records_rejected},

            -- Store the technical failure reason.
            error_message = '{error_message.replace("'", "''")}'

        WHERE
            run_id = '{run_id}'
            AND source_file = '{source_file}'
            AND status = 'PROCESSING'
    """)


    # Confirm that the file failure was recorded.
    print(
        f"File processing marked as FAILED: "
        f"{source_file}"
    )


    # Return the failure timestamp.
    return processing_end_time

Create the technical error logging function

In [0]:
# ============================================================
# CELL 33 — CREATE TECHNICAL ERROR LOGGING FUNCTION
# ============================================================

# Define a reusable function for recording technical errors.
#
# This is separate from Data Quality failures.
# Examples of technical errors:
# - File cannot be read
# - Table does not exist
# - Schema error
# - Permission error
# - Spark runtime failure
def log_pipeline_error(
    run_id,
    pipeline_id,
    task_name,
    error_type,
    error_message,
    stack_trace=None
):
    """
    Write a technical error into pipeline_error_log.

    Parameters:
        run_id:
            Unique identifier of the pipeline execution.

        pipeline_id:
            Pipeline configuration identifier.

        task_name:
            Name of the task where the error occurred.

        error_type:
            Category of the technical error.

        error_message:
            Human-readable error message.

        stack_trace:
            Optional detailed technical stack trace.
    """

    # Capture the time when the technical error occurred.
    error_time = spark.sql(
        "SELECT current_timestamp() AS current_time"
    ).collect()[0]["current_time"]


    # Escape single quotes in the error message.
    # This prevents SQL syntax problems when we insert
    # an error containing an apostrophe.
    safe_error_message = (
        error_message.replace("'", "''")
    )


    # Escape single quotes in the stack trace when present.
    if stack_trace is not None:

        safe_stack_trace = (
            stack_trace.replace("'", "''")
        )

    else:

        safe_stack_trace = None


    # Build the stack-trace SQL value.
    if safe_stack_trace is None:

        stack_trace_sql = "NULL"

    else:

        stack_trace_sql = (
            f"'{safe_stack_trace}'"
        )


    # Insert the technical error into the audit table.
    spark.sql(f"""
        INSERT INTO workspace.nyc_taxi_audit.pipeline_error_log
        VALUES (
            '{run_id}',
            '{pipeline_id}',
            '{task_name}',
            '{error_type}',
            '{safe_error_message}',
            {stack_trace_sql},
            TIMESTAMP('{error_time}')
        )
    """)


    # Confirm that the technical error was logged.
    print(
        f"Technical error logged successfully. "
        f"Run ID: {run_id}, "
        f"Task: {task_name}, "
        f"Error type: {error_type}"
    )

Create task audit function

In [0]:
# ============================================================
# CELL 34 — CREATE TASK AUDIT FUNCTION
# ============================================================

# Define a reusable function for recording task execution.
#
# Our pipeline can contain multiple logical tasks, for example:
#
# 1. File discovery
# 2. Source read
# 3. Record ID generation
# 4. Bronze write
#
# This function allows us to track each task independently.
def log_task_status(
    run_id,
    pipeline_id,
    task_name,
    task_sequence,
    start_time,
    end_time,
    status,
    records_processed=0,
    error_message=None,
    task_run_id=None
):
    """
    Write one task execution record to pipeline_task_log.

    Parameters:
        run_id:
            Unique identifier for the pipeline execution.

        pipeline_id:
            Pipeline configuration identifier.

        task_name:
            Name of the task being tracked.

        task_sequence:
            Execution order of the task.

        start_time:
            Task start timestamp.

        end_time:
            Task completion timestamp.

        status:
            Task status such as SUCCESS or FAILED.

        records_processed:
            Number of records processed by the task.

        error_message:
            Optional task error message.

        task_run_id:
            Databricks task identifier when available.
    """

    # Use the task end time as the audit-record creation time
    # when an explicit creation timestamp is not required.
    created_at = (
        end_time
        if end_time is not None
        else start_time
    )


    # Escape single quotes in the error message so that
    # the generated SQL remains valid.
    if error_message is not None:

        safe_error_message = (
            error_message.replace("'", "''")
        )

        error_sql = (
            f"'{safe_error_message}'"
        )

    else:

        error_sql = "NULL"


    # Escape a task run ID when one is supplied.
    if task_run_id is not None:

        safe_task_run_id = (
            task_run_id.replace("'", "''")
        )

        task_run_id_sql = (
            f"'{safe_task_run_id}'"
        )

    else:

        task_run_id_sql = "NULL"


    # Insert the task execution record into the audit table.
    spark.sql(f"""
        INSERT INTO workspace.nyc_taxi_audit.pipeline_task_log
        VALUES (
            '{run_id}',
            '{pipeline_id}',
            '{task_name}',
            {task_sequence},
            TIMESTAMP('{start_time}'),
            {f"TIMESTAMP('{end_time}')" if end_time is not None else "NULL"},
            '{status}',
            {records_processed},
            {error_sql},
            {task_run_id_sql},
            TIMESTAMP('{created_at}')
        )
    """)


    # Confirm that the task status was recorded.
    print(
        f"Task audit recorded: "
        f"{task_name} → {status}"
    )

Process only eligible files

In [0]:
# ============================================================
# CELL 35 — PROCESS ELIGIBLE FILES
# ============================================================

# Check whether any files are eligible for processing.
#
# If the list is empty, the pipeline should perform no
# data processing. This protects us from reprocessing files
# that were already marked as SUCCESS.
if not new_file_paths:

    # Display a clear no-op message.
    print(
        "No new files are eligible for processing."
    )

    print(
        "All discovered files have already been "
        "successfully processed."
    )


# If one or more new files are available, process them.
else:

    # Display the number of files that will be processed.
    print(
        f"Files eligible for processing: "
        f"{len(new_file_paths)}"
    )


    # Process each eligible file one at a time.
    #
    # We use a loop because our framework is designed to
    # support multiple monthly source files.
    for file_path in new_file_paths:

        # Display the file currently being processed.
        print(
            f"Processing file: {file_path}"
        )

Create the Bronze processing function

In [0]:
# ============================================================
# CELL 36 — CREATE BRONZE FILE PROCESSING FUNCTION
# ============================================================

# Define a reusable function that processes one source file.
#
# The function receives the pipeline configuration and the
# source file path instead of using hardcoded dataset values.
def process_bronze_file(
    file_path,
    pipeline_config,
    run_id
):
    """
    Process one source file into the Bronze Delta table.

    Parameters:
        file_path:
            Complete path of the source file.

        pipeline_config:
            Metadata-driven configuration dictionary.

        run_id:
            Unique ID of the current pipeline execution.

    Returns:
        Dictionary containing processing metrics.
    """

    # --------------------------------------------------------
    # 1. READ FILE-LEVEL CONFIGURATION
    # --------------------------------------------------------

    # Read the source format from metadata.
    source_format = (
        pipeline_config["source_format"]
    )

    # Read the target table from metadata.
    target_table = (
        f"{pipeline_config['target_catalog']}."
        f"{pipeline_config['target_schema']}."
        f"{pipeline_config['target_table']}"
    )

    # Read the configured hash-column string.
    hash_columns_string = (
        pipeline_config["record_hash_columns"]
    )


    # Convert the comma-separated hash columns into
    # a Python list.
    hash_columns = [
        column_name.strip()
        for column_name in hash_columns_string.split(",")
        if column_name.strip()
    ]


    # --------------------------------------------------------
    # 2. VALIDATE SOURCE FORMAT
    # --------------------------------------------------------

    # Our current framework supports Parquet.
    if source_format.lower() != "parquet":

        raise ValueError(
            f"Unsupported source format: {source_format}"
        )


    # --------------------------------------------------------
    # 3. START FILE-LEVEL AUDIT
    # --------------------------------------------------------

    # Find the FileInfo object corresponding to this path.
    matching_file_info = next(
        (
            file_info
            for file_info in matching_files
            if file_info.path == file_path
        ),
        None
    )


    # Make sure file metadata could be found.
    if matching_file_info is None:

        raise FileNotFoundError(
            f"File metadata not found for: {file_path}"
        )


    # Record this file as PROCESSING.
    file_processing_start = start_file_processing(
        run_id=run_id,
        pipeline_id=pipeline_config["pipeline_id"],
        pipeline_name=pipeline_config["pipeline_name"],
        file_info=matching_file_info
    )


    # --------------------------------------------------------
    # 4. READ SOURCE FILE
    # --------------------------------------------------------

    try:

        # Read the source Parquet file.
        source_df = spark.read.parquet(
            file_path
        )


        # Count the records read from the source.
        records_read = source_df.count()


        # ----------------------------------------------------
        # 5. GENERATE DETERMINISTIC RECORD ID
        # ----------------------------------------------------

        # Generate a SHA-256 record ID from the configured
        # source columns.
        bronze_file_df = (
            source_df
            .withColumn(
                "record_id",
                F.sha2(
                    F.concat_ws(
                        "||",
                        *[
                            F.coalesce(
                                F.col(column_name).cast("string"),
                                F.lit("<NULL>")
                            )
                            for column_name in hash_columns
                        ]
                    ),
                    256
                )
            )
        )


        # ----------------------------------------------------
        # 6. ADD INGESTION METADATA
        # ----------------------------------------------------

        # Add the current pipeline run ID.
        bronze_file_df = bronze_file_df.withColumn(
            "run_id",
            F.lit(run_id)
        )


        # Add the source file path.
        bronze_file_df = bronze_file_df.withColumn(
            "source_file",
            F.lit(file_path)
        )


        # Add the ingestion timestamp.
        bronze_file_df = bronze_file_df.withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )


        # ----------------------------------------------------
        # 7. VALIDATE RECORD ID
        # ----------------------------------------------------

        # Count unique record IDs.
        distinct_record_ids = (
            bronze_file_df
            .select("record_id")
            .distinct()
            .count()
        )


        # Calculate duplicate record IDs.
        duplicate_record_ids = (
            records_read - distinct_record_ids
        )


        # Stop processing if duplicate technical IDs exist.
        if duplicate_record_ids > 0:

            raise ValueError(
                f"Duplicate record_id values detected: "
                f"{duplicate_record_ids}"
            )


        # ----------------------------------------------------
        # 8. WRITE TO BRONZE
        # ----------------------------------------------------

        # Append the processed file to the Bronze table.
        #
        # IMPORTANT:
        # File-level idempotency has already been checked before
        # this function is called.
        bronze_file_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(
                target_table
            )


        # Count the records after the write.
        records_written = records_read


        # No DQ rejection occurs in Bronze at this stage.
        records_rejected = 0


        # ----------------------------------------------------
        # 9. COMPLETE FILE AUDIT
        # ----------------------------------------------------

        # Mark the file as successfully processed.
        complete_file_processing(
            run_id=run_id,
            source_file=file_path,
            records_read=records_read,
            records_written=records_written,
            records_rejected=records_rejected
        )


        # ----------------------------------------------------
        # 10. RETURN PROCESSING METRICS
        # ----------------------------------------------------

        return {
            "source_file": file_path,
            "records_read": records_read,
            "records_written": records_written,
            "records_rejected": records_rejected,
            "status": "SUCCESS"
        }


    # --------------------------------------------------------
    # 11. HANDLE TECHNICAL FAILURE
    # --------------------------------------------------------

    except Exception as error:

        # Capture the complete error message.
        error_message = str(error)


        # Mark the file as FAILED in file_ingestion_log.
        fail_file_processing(
            run_id=run_id,
            source_file=file_path,
            error_message=error_message,
            records_read=locals().get(
                "records_read",
                0
            ),
            records_written=locals().get(
                "records_written",
                0
            ),
            records_rejected=locals().get(
                "records_rejected",
                0
            )
        )


        # Log the technical error separately.
        log_pipeline_error(
            run_id=run_id,
            pipeline_id=pipeline_config["pipeline_id"],
            task_name="bronze_file_processing",
            error_type="BRONZE_INGESTION_ERROR",
            error_message=error_message
        )


        # Re-raise the exception so the calling pipeline
        # knows that processing failed.
        raise

Test the function safely

In [0]:
# ============================================================
# CELL 37 — VERIFY BRONZE PROCESSING FUNCTION
# ============================================================

# Check that the reusable Bronze processing function
# has been successfully defined in the notebook.
if not callable(process_bronze_file):

    # Stop execution if the function is not available.
    raise ValueError(
        "process_bronze_file function is not available."
    )


# Confirm that the current list of new files is available.
if new_file_paths is None:

    # Stop execution if the processing list was not created.
    raise ValueError(
        "new_file_paths is not available."
    )


# Display the number of files currently eligible for processing.
print(
    f"New files available for processing: "
    f"{len(new_file_paths)}"
)


# Confirm that the reusable processing function
# is ready for the next ingestion cycle.
print(
    "Bronze processing function is ready."
)

Create the ingestion controller function

In [0]:
# ============================================================
# CELL 38 — CREATE BRONZE INGESTION CONTROLLER
# ============================================================

# Define a reusable controller function.
# Its responsibility is to coordinate the processing of
# all files that are eligible for ingestion.
def run_bronze_ingestion(
    new_file_paths,
    pipeline_config,
    run_id
):
    """
    Process all eligible source files.

    Parameters:
        new_file_paths:
            List of source files that have not been
            successfully processed before.

        pipeline_config:
            Metadata-driven pipeline configuration.

        run_id:
            Unique identifier for the current pipeline run.

    Returns:
        List of processing results for each file.
    """

    # Create an empty list to hold the result returned
    # from processing each source file.
    processing_results = []


    # Check whether there is any new work to process.
    if not new_file_paths:

        # Return an empty result when there is no new data.
        print(
            "No new files available for Bronze ingestion."
        )

        return processing_results


    # Display the number of files that will be processed.
    print(
        f"Starting Bronze ingestion for "
        f"{len(new_file_paths)} file(s)."
    )


    # Process each eligible file independently.
    for file_path in new_file_paths:

        # Display the file currently being processed.
        print(
            f"Starting file: {file_path}"
        )


        # Find the Databricks FileInfo object corresponding
        # to the current source file.
        file_info = next(
            (
                file
                for file in matching_files
                if file.path == file_path
            ),
            None
        )


        # Make sure the file metadata exists.
        if file_info is None:

            # Stop processing because the file cannot be
            # properly audited without its metadata.
            raise FileNotFoundError(
                f"File metadata not found for: {file_path}"
            )


        # Call our reusable Bronze file-processing function.
        result = process_bronze_file(
            file_path=file_path,
            pipeline_config=pipeline_config,
            run_id=run_id
        )


        # Store the result so the caller can use the
        # processing metrics later.
        processing_results.append(
            result
        )


        # Display the result for the processed file.
        print(
            f"Completed file: {file_path}"
        )

        print(
            f"Status: {result['status']}"
        )


    # Return the results for all processed files.
    return processing_results

Validate Bronze target readiness

In [0]:
# ============================================================
# CELL 39 — VALIDATE BRONZE TARGET READINESS
# ============================================================

# Check whether the target Bronze table currently exists.
# The fully qualified table name comes from metadata.
target_table_exists = spark.catalog.tableExists(
    target_table_name
)


# Validate that the target table is available.
if not target_table_exists:

    # Stop execution because our current processing function
    # is designed to append into an existing Bronze table.
    raise ValueError(
        f"Bronze target table does not exist: "
        f"{target_table_name}"
    )


# Confirm that the target table is ready.
print(
    f"Bronze target table is ready: "
    f"{target_table_name}"
)

Run the Bronze controller

In [0]:
# ============================================================
# CELL 40 — RUN BRONZE INGESTION CONTROLLER
# ============================================================

# Call the reusable Bronze ingestion controller.
#
# The controller receives:
#   1. new_file_paths
#      → only files that have not already succeeded
#
#   2. pipeline_config
#      → configuration obtained from metadata
#
#   3. run_id
#      → identifier for this pipeline execution
#
# The controller then calls process_bronze_file()
# for every eligible file.
processing_results = run_bronze_ingestion(
    new_file_paths=new_file_paths,
    pipeline_config=pipeline_config,
    run_id=run_id
)


# ============================================================
# DISPLAY EXECUTION SUMMARY
# ============================================================

# Check whether the controller found any files to process.
if not processing_results:

    # No new files means this execution performed no ingestion.
    print(
        "Bronze ingestion completed with no new files."
    )

else:

    # Display the number of files actually processed.
    print(
        f"Bronze ingestion processed "
        f"{len(processing_results)} file(s)."
    )


# Display each file-processing result.
for result in processing_results:

    print(
        f"File: {result['source_file']}"
    )

    print(
        f"Status: {result['status']}"
    )

    print(
        f"Records read: {result['records_read']}"
    )

    print(
        f"Records written: {result['records_written']}"
    )

    print(
        f"Records rejected: {result['records_rejected']}"
    )

Verify idempotency after rerun

In [0]:
# ============================================================
# CELL 41 — VERIFY BRONZE COUNT AFTER RERUN
# ============================================================

# Read the Bronze target table again.
# We want to verify that the rerun did not append duplicate data.
bronze_after_rerun_df = spark.table(
    target_table_name
)


# Count the records currently stored in Bronze.
bronze_count_after_rerun = (
    bronze_after_rerun_df.count()
)


# Display the Bronze count after the rerun.
print(
    f"Bronze record count after rerun: "
    f"{bronze_count_after_rerun}"
)


# Display the baseline count that we captured before
# testing the rerun.
print(
    f"Bronze record count before rerun: "
    f"{bronze_count_before_rerun}"
)


# Compare the two counts.
#
# For an idempotent rerun, these values must be identical.
if bronze_count_after_rerun != bronze_count_before_rerun:

    # Stop execution if the count changed unexpectedly.
    raise ValueError(
        "IDEMPOTENCY VALIDATION FAILED: "
        f"Bronze count changed from "
        f"{bronze_count_before_rerun} to "
        f"{bronze_count_after_rerun}."
    )


# Confirm that the rerun did not change the Bronze record count.
print(
    "IDEMPOTENCY VALIDATION PASSED: "
    "Bronze record count is unchanged."
)

Validate Bronze record lineage

In [0]:
# ============================================================
# CELL 42 — VALIDATE BRONZE RECORD LINEAGE
# ============================================================

# Define the technical lineage columns that every Bronze
# record is expected to contain.
required_lineage_columns = [
    "record_id",
    "run_id",
    "source_file",
    "ingestion_timestamp"
]


# Get the actual columns available in the Bronze table.
bronze_columns = set(
    bronze_after_rerun_df.columns
)


# Identify any required lineage columns that are missing.
missing_lineage_columns = [
    column_name
    for column_name in required_lineage_columns
    if column_name not in bronze_columns
]


# Stop the pipeline if any required lineage columns
# are missing from the Bronze table.
if missing_lineage_columns:

    raise ValueError(
        "Missing Bronze lineage columns: "
        f"{missing_lineage_columns}"
    )


# Check for records where any required lineage column
# is NULL.
null_lineage_condition = None

for column_name in required_lineage_columns:

    # Build the NULL condition for the current column.
    current_condition = (
        F.col(column_name).isNull()
    )

    # Combine conditions for all lineage columns.
    if null_lineage_condition is None:

        null_lineage_condition = current_condition

    else:

        null_lineage_condition = (
            null_lineage_condition
            | current_condition
        )


# Count records containing NULL lineage values.
null_lineage_records = (
    bronze_after_rerun_df
    .filter(null_lineage_condition)
    .count()
)


# Display the validation result.
print(
    f"Bronze records with NULL lineage values: "
    f"{null_lineage_records}"
)


# Fail validation if any Bronze record has incomplete lineage.
if null_lineage_records > 0:

    raise ValueError(
        f"Bronze lineage validation failed. "
        f"Records with NULL lineage: "
        f"{null_lineage_records}"
    )


# Confirm successful lineage validation.
print(
    "Bronze lineage validation successful."
)

print(
    "Required lineage columns:",
    required_lineage_columns
)